# 🚀 Forest Inspection - Training Notebook

Train semantic segmentation models trên Kaggle GPU.

**Models**: U-Net, HRNet, DeepLabV3+, PointFlow

In [ ]:
# Install dependencies (Kaggle)
# !pip install segmentation-models-pytorch albumentations omegaconf torchmetrics -q

In [ ]:
import sys
import torch
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.insert(0, '.')

from src.data.dataset import ForestDataset, NUM_CLASSES
from src.data.transforms import get_train_transforms, get_val_transforms
from src.data.splits import get_split, print_split_info
from src.models import build_model
from src.training.losses import CEDiceLoss
from src.training.trainer import Trainer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## 1. Configuration

In [ ]:
# ===== CONFIGURATION =====
# DATA_ROOT = '/kaggle/input/forest-sunny'  # Kaggle
DATA_ROOT = 'data/forest_sunny'  # Local

MODEL_NAME = 'unet'          # unet, hrnet, deeplabv3plus, pointflow
ENCODER = 'resnet34'         # resnet34, resnet50, tu-hrnet_w18
IMG_SIZE = (512, 512)
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

print(f'Model: {MODEL_NAME} ({ENCODER})')
print(f'Image size: {IMG_SIZE}, Batch: {BATCH_SIZE}, Epochs: {EPOCHS}')

## 2. Data Loading

In [ ]:
# Split
split = get_split('cross_sequence')
print_split_info(split)

# Transforms
train_tf = get_train_transforms(img_size=IMG_SIZE)
val_tf = get_val_transforms(img_size=IMG_SIZE)

# Datasets
train_ds = ForestDataset(DATA_ROOT, split['train'], transform=train_tf)
val_ds = ForestDataset(DATA_ROOT, split['val'], transform=val_tf)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nTrain: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples, {len(val_loader)} batches')

In [ ]:
# Sanity check: visualize a batch
import matplotlib.pyplot as plt
from src.data.dataset import class_id_to_rgb

images, masks = next(iter(train_loader))
print(f'Batch shape: images={images.shape}, masks={masks.shape}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = images[i].permute(1,2,0).numpy()
    img = (img * [0.229,0.224,0.225] + [0.485,0.456,0.406]) * 255
    axes[0,i].imshow(img.clip(0,255).astype('uint8'))
    axes[0,i].set_title(f'Image {i}')
    axes[0,i].axis('off')
    
    lbl_rgb = class_id_to_rgb(masks[i].numpy())
    axes[1,i].imshow(lbl_rgb)
    axes[1,i].set_title(f'Label {i}')
    axes[1,i].axis('off')
plt.tight_layout()
plt.show()

## 3. Model & Training

In [ ]:
# Build model
model = build_model(MODEL_NAME, encoder=ENCODER, num_classes=NUM_CLASSES)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME} ({ENCODER})')
print(f'Parameters: {total_params:,}')

In [ ]:
# Loss, optimizer, scheduler
criterion = CEDiceLoss(num_classes=NUM_CLASSES, ce_weight=1.0, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config={
        'amp': True,
        'accumulation_steps': 1,
        'log_dir': 'outputs/logs',
        'checkpoint_dir': 'outputs/checkpoints',
        'save_top_k': 3,
        'early_stopping': {'patience': 15},
        'num_classes': NUM_CLASSES,
        'log_every_n_steps': 10,
    },
    device=device,
)

# Train!
best_results = trainer.fit(num_epochs=EPOCHS)

In [ ]:
# Print results
print('\n📊 Best Results:')
print(f"  mIoU: {best_results.get('miou', 0):.4f}")
print(f"  Pixel Accuracy: {best_results.get('pixel_accuracy', 0):.4f}")
for name in ['Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees', 'Ground_vegetation']:
    print(f"  IoU {name}: {best_results.get(f'iou_{name}', 0):.4f}")